# Multiome1 Human: base-pretrained vs LoRA-finetuned on held-out neurons

Loads both prediction zarrs, collapses to one (observed, predicted) value per gene, and compares the base pretrained model with the LoRA fine-tuned model.

Run after inference has produced:

- `$GET_COURSE_WORK/output/finetune_multiome1_human/interpret_base_neurons/neurons.zarr`
- `$GET_COURSE_WORK/output/finetune_multiome1_human/interpret_ft_neurons/neurons.zarr`

Inside the container, for example:

```bash
cd ~/AI_in_genomics/GET
apptainer exec --nv     -B $PWD:$PWD     -B $GET_COURSE_WORK:$GET_COURSE_WORK     $GET_SIF     bash -lc "PYTHONNOUSERSITE=1 jupyter nbconvert --to notebook --execute tutorials/compare_base_vs_finetuned.ipynb --output compare_base_vs_finetuned.executed.ipynb"
```

The executed notebook is a generated artifact and should not be committed.


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zarr
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, r2_score

COURSE_WORK = Path(os.environ.get("GET_COURSE_WORK", Path.home() / "GET_course_work"))
OUT_ROOT = Path(
    os.environ.get(
        "GET_MULTIOME_OUTPUT_ROOT",
        COURSE_WORK / "output" / "finetune_multiome1_human",
    )
)
BASE_ZARR = Path(os.environ.get("GET_BASE_INTERPRET_ZARR", OUT_ROOT / "interpret_base_neurons" / "neurons.zarr"))
FT_ZARR = Path(os.environ.get("GET_FT_INTERPRET_ZARR", OUT_ROOT / "interpret_ft_neurons" / "neurons.zarr"))
FIG_DIR = Path(os.environ.get("GET_COMPARE_OUT_DIR", OUT_ROOT / "compare_base_vs_finetuned"))
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("base:", BASE_ZARR, BASE_ZARR.exists())
print("ft  :", FT_ZARR, FT_ZARR.exists())


In [ ]:
def load_per_gene(zarr_path: Path) -> pd.DataFrame:
    """Collapse the (n_samples, 200, 2) obs/pred blocks to one value per gene.

    For each sample:
      - strand picks the correct column in the last axis
      - `focus` identifies the regions inside the 200-peak window that are
        the TSS for the gene (values >=0 are valid peak indices; -1 is pad)
      - we take the max over those TSS peaks, matching how log-TPM is
        defined on the max signal of the TSS region set.
    When a gene has multiple samples (multi-TSS genes), we take the max
    across samples as well.
    """
    z = zarr.open(str(zarr_path), mode="r")
    preds = z["preds"]["exp"][:]        # (N, 200, 2)
    obs   = z["obs"]["exp"][:]           # (N, 200, 2)
    strand = z["strand"][:].astype(int)  # (N,)
    focus = z["focus"][:]                # (N, 200)
    genes = np.array([g.strip() for g in z["available_genes"][:]])
    chroms = np.array([c.strip() for c in z["chromosome"][:]])

    n = preds.shape[0]
    rows = []
    for i in range(n):
        peaks = focus[i]
        valid = peaks[peaks >= 0].astype(int)
        if valid.size == 0:
            continue
        s = strand[i]
        p = preds[i, valid, s].max()
        o = obs[i, valid, s].max()
        rows.append((genes[i], chroms[i], int(s), float(o), float(p)))
    df = pd.DataFrame(rows, columns=["gene", "chrom", "strand", "obs", "pred"])
    df = df.groupby("gene", as_index=False).agg(
        chrom=("chrom", "first"),
        strand=("strand", "first"),
        obs=("obs", "max"),
        pred=("pred", "max"),
    )
    return df


base_df = load_per_gene(BASE_ZARR)
ft_df   = load_per_gene(FT_ZARR)
print('base per-gene table:', base_df.shape)
print('ft   per-gene table:', ft_df.shape)
base_df.head()

In [ ]:
def metrics(df: pd.DataFrame) -> dict:
    x = df["obs"].to_numpy()
    y = df["pred"].to_numpy()
    r, _  = pearsonr(x, y)
    rho, _ = spearmanr(x, y)
    mse = mean_squared_error(x, y)
    fve = 1.0 - np.var(x - y) / np.var(x)   # fraction of variance explained
    r2  = r2_score(x, y)
    return {
        "n": len(df),
        "pearson_r": r,
        "spearman_rho": rho,
        "mse": mse,
        "var_explained": fve,
        "r2": r2,
    }


summary = pd.DataFrame(
    {
        "base_pretrained": metrics(base_df),
        "lora_finetuned":  metrics(ft_df),
    }
).T
summary

In [ ]:
merged = pd.merge(
    base_df.rename(columns={"pred": "pred_base"})[["gene", "chrom", "obs", "pred_base"]],
    ft_df.rename(columns={"pred": "pred_ft"})[["gene", "pred_ft"]],
    on="gene",
    how="inner",
)
print('merged shape:', merged.shape)
merged.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

def _scatter(ax, x, y, title):
    ax.scatter(x, y, s=4, alpha=0.3, edgecolor="none")
    lim_lo = min(x.min(), y.min())
    lim_hi = max(x.max(), y.max())
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], "k--", lw=0.8)
    r, _ = pearsonr(x, y)
    rho, _ = spearmanr(x, y)
    ax.set_xlabel("observed log10 TPM+1")
    ax.set_ylabel("predicted")
    ax.set_title(f"{title}\nPearson r={r:.3f}  Spearman ρ={rho:.3f}  n={len(x)}")

_scatter(axes[0], merged["obs"].to_numpy(), merged["pred_base"].to_numpy(), "neurons: base pretrained")
_scatter(axes[1], merged["obs"].to_numpy(), merged["pred_ft"].to_numpy(),   "neurons: LoRA finetuned")

# third panel: pred_ft vs pred_base (how different are the two models)
axes[2].scatter(merged["pred_base"], merged["pred_ft"], s=4, alpha=0.3, edgecolor="none")
lo = min(merged["pred_base"].min(), merged["pred_ft"].min())
hi = max(merged["pred_base"].max(), merged["pred_ft"].max())
axes[2].plot([lo, hi], [lo, hi], "k--", lw=0.8)
axes[2].set_xlabel("base pretrained")
axes[2].set_ylabel("LoRA finetuned")
axes[2].set_title("prediction delta: FT vs base")

plt.tight_layout()
fig.savefig(FIG_DIR / "scatter_base_vs_ft.png", dpi=150)
plt.show()

In [ ]:
# bar summary of metrics
fig, ax = plt.subplots(1, 1, figsize=(6.5, 3.8))
metric_names = ["pearson_r", "spearman_rho", "var_explained"]
vals_base = [metrics(base_df)[m] for m in metric_names]
vals_ft   = [metrics(ft_df)[m]   for m in metric_names]
xs = np.arange(len(metric_names))
w = 0.38
ax.bar(xs - w/2, vals_base, width=w, label="base pretrained", color="#888")
ax.bar(xs + w/2, vals_ft,   width=w, label="LoRA finetuned", color="#2c7bb6")
ax.set_xticks(xs)
ax.set_xticklabels(metric_names)
ax.axhline(0, color="k", lw=0.6)
ax.legend()
ax.set_title("Per-gene expression metrics on held-out neurons")
plt.tight_layout()
fig.savefig(FIG_DIR / "metrics_bar.png", dpi=150)
plt.show()

In [ ]:
# Per-chromosome breakdown of Pearson r for both models.
rows = []
for chrom, sub in merged.groupby("chrom"):
    if len(sub) < 20:
        continue
    rows.append({
        "chrom": chrom,
        "n": len(sub),
        "r_base": pearsonr(sub["obs"], sub["pred_base"])[0],
        "r_ft":   pearsonr(sub["obs"], sub["pred_ft"])[0],
    })
by_chrom = pd.DataFrame(rows).sort_values("chrom")
by_chrom.to_csv(FIG_DIR / "per_chromosome_pearson.csv", index=False)
by_chrom

In [ ]:
merged.to_csv(FIG_DIR / "per_gene_predictions.csv", index=False)
summary.to_csv(FIG_DIR / "summary_metrics.csv")
print('wrote:', FIG_DIR)

## Optional training-length comparison (held-out neurons)

Snapshot from the previous smoke run (`archive_10epoch/summary_metrics.csv` and `per_chromosome_pearson.csv`) vs the current fine-tuned model metrics computed above.

In [ ]:
ARCHIVE = OUT_ROOT / "archive_10epoch"
prev_summary = pd.read_csv(ARCHIVE / "summary_metrics.csv", index_col=0)
prev_chrom = pd.read_csv(ARCHIVE / "per_chromosome_pearson.csv")

ft_10 = prev_summary.loc["lora_finetuned"]
ft_100 = summary.loc["lora_finetuned"]

delta = pd.DataFrame(
    {
        "metric": ["pearson_r", "spearman_rho", "mse", "var_explained", "r2"],
        "lora_10ep": [ft_10[m] for m in ["pearson_r", "spearman_rho", "mse", "var_explained", "r2"]],
        "lora_100ep": [ft_100[m] for m in ["pearson_r", "spearman_rho", "mse", "var_explained", "r2"]],
    }
)
delta["delta_100_minus_10"] = delta["lora_100ep"] - delta["lora_10ep"]
print(delta.to_string(index=False))

chrom_cmp = by_chrom.merge(
    prev_chrom[["chrom", "r_ft"]].rename(columns={"r_ft": "r_ft_10ep"}),
    on="chrom",
    how="inner",
)
chrom_cmp["delta_r_ft"] = chrom_cmp["r_ft"] - chrom_cmp["r_ft_10ep"]
chrom_cmp.to_csv(FIG_DIR / "per_chromosome_pearson_10ep_vs_100ep.csv", index=False)

fig, ax = plt.subplots(1, 1, figsize=(6.5, 3.8))
metric_plot = ["pearson_r", "spearman_rho", "var_explained"]
x = np.arange(len(metric_plot))
w = 0.35
v10 = [ft_10[m] for m in metric_plot]
v100 = [ft_100[m] for m in metric_plot]
ax.bar(x - w / 2, v10, width=w, label="LoRA 10 ep (archive)", color="#fdae61")
ax.bar(x + w / 2, v100, width=w, label="LoRA 100 ep (this run)", color="#2c7bb6")
ax.set_xticks(x)
ax.set_xticklabels(metric_plot)
ax.axhline(0, color="k", lw=0.6)
ax.legend()
ax.set_title("Fine-tuned model: prior 10-epoch vs current 100-epoch run")
plt.tight_layout()
fig.savefig(FIG_DIR / "metrics_bar_10ep_vs_100ep_ft.png", dpi=150)
plt.show()